## 11. Building Prediction Pipeline and Model Deployment

## 11.1 Introduction

This section focuses on developing a prediction pipeline that connects the trained Student Performance Prediction model to a user-facing application. The pipeline will accept student academic information, process the inputs using the saved machine learning pipeline, and generate predictions for the student's final mathematics grade (G3).


## 11.2 Understanding the Prediction Inputs

The prediction system uses four input features selected during the model development stage:

| Feature | Description |
|---|---|
| studytime | Weekly study time category (1–4) |
| absences | Number of school absences |
| G1 | First-period mathematics grade |
| G2 | Second-period mathematics grade |

The target variable is `G3`, representing the student's final mathematics grade on a scale of 0 to 20.

These features must be provided using the same definitions and numerical representation used during model training.



## 11.3 Testing the Prediction Pipeline

Before integrating the prediction pipeline into a Flask application, we first test whether it can successfully accept student information and generate a prediction using the saved model.


In [2]:
import os
import sys

# Set the project root directory
project_root = r"C:\Users\helle\Desktop\Predictive-analytics-project"

# Change the working directory
os.chdir(project_root)

# Add project root to Python's import path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify the working directory
print("Working directory:", os.getcwd())
print("src folder exists:", os.path.isdir("src"))

Working directory: C:\Users\helle\Desktop\Predictive-analytics-project
src folder exists: True


In [3]:
from src.pipeline.predict_pipeline import CustomData, PredictPipeline

print("Prediction pipeline imported successfully!")

Prediction pipeline imported successfully!


In [4]:
# Create sample student information
student_data = CustomData(
    studytime=3,
    absences=2,
    G1=12,
    G2=14
)

# Convert the information into a DataFrame
student_df = student_data.to_dataframe()

print("Student Information:")
display(student_df)

# Generate the prediction
predict_pipeline = PredictPipeline()
prediction = predict_pipeline.predict(student_df)

print("Predicted Final Mathematics Grade (G3):", round(prediction, 2))

Student Information:


,studytime,absences,G1,G2
0,3.0,2.0,12.0,14.0


Predicted Final Mathematics Grade (G3): 13.95



### 11.4 Prediction Pipeline Testing Results

The prediction pipeline was successfully tested using sample student academic information.

The test confirmed that:

- The CustomData class correctly converts student information into a Pandas DataFrame.
- The PredictPipeline class successfully loads the saved XGBoost model.
- The model generates a numerical prediction using the provided student information.

**Sample Prediction Result:**

For a student with studytime = 3, absences = 2, G1 = 12 and G2 = 14, the model predicted a final mathematics grade (G3) of 13.95 out of 20.

The successful test confirms that the prediction pipeline is functioning as expected and is ready for integration into a web application.


In [5]:
from app import app

# Create a Flask test client
client = app.test_client()

# Submit invalid student information directly
response = client.post("/", data={
    "studytime": "2",
    "absences": "-9",
    "G1": "10",
    "G2": "15"
})

# Display the response
print("Response status:", response.status_code)

if b"Number of absences cannot be negative." in response.data:
    print("Validation successful: Negative absences rejected.")
else:
    print("Validation message not found.")

Response status: 200
Validation successful: Negative absences rejected.


In [6]:
test_cases = [
    {
        "name": "Invalid study time",
        "data": {
            "studytime": "5",
            "absences": "2",
            "G1": "10",
            "G2": "15"
        },
        "expected": "Study time must be selected"
    },
    {
        "name": "G1 above 20",
        "data": {
            "studytime": "2",
            "absences": "2",
            "G1": "25",
            "G2": "15"
        },
        "expected": "First period grade (G1) must be between 0 and 20"
    },
    {
        "name": "G2 below 0",
        "data": {
            "studytime": "2",
            "absences": "2",
            "G1": "10",
            "G2": "-2"
        },
        "expected": "Second period grade (G2) must be between 0 and 20"
    },
    {
        "name": "Missing G1",
        "data": {
            "studytime": "2",
            "absences": "2",
            "G1": "",
            "G2": "15"
        },
        "expected": "Please provide a value for G1"
    }
]

for test in test_cases:

    response = client.post("/", data=test["data"])

    passed = test["expected"].encode() in response.data

    print(
        f"{test['name']}: "
        f"{'PASSED' if passed else 'FAILED'}"
    )

Invalid study time: PASSED
G1 above 20: PASSED
G2 below 0: PASSED
Missing G1: PASSED



## 11.5 Input Validation and Error Handling

Input validation was implemented in the Flask application to ensure that the student information submitted through the web form meets the required conditions before being passed to the prediction pipeline.

Two levels of validation were incorporated:

1. **Client-side validation:** HTML input attributes help prevent invalid entries through the browser.
2. **Server-side validation:** Python validation functions independently verify submitted values before model prediction.

The following test cases were conducted:

| Test case | Expected behaviour | Result |
|---|---|---|
| Negative absences | Reject input | Passed |
| Invalid study time | Reject input | Passed |
| G1 above 20 | Reject input | Passed |
| G2 below 0 | Reject input | Passed |
| Missing G1 | Reject input | Passed |

All validation tests passed successfully.

This ensures that invalid student information is rejected before reaching the trained machine learning model.


## 11.6 Deployment Configuration

After successfully testing the prediction pipeline locally, the Flask application was prepared for deployment so that users could access the student performance prediction system through a web browser.

The application was deployed using **Render**, a cloud platform that supports hosting Python web applications.

The following configurations were used:

| Configuration | Setting |
|---|---|
| Hosting Platform | Render |
| Application Framework | Flask |
| Deployment Branch | feature/data_preprocessing |
| Python Runtime | Python |
| Build Command | `pip install -r requirements.txt` |
| Start Command | `gunicorn --workers 1 --bind 0.0.0.0:$PORT app:app` |
| Instance Type | Free |
| Auto-Deploy | On Commit |

The deployment process involved connecting the GitHub repository to Render, configuring the build and start commands, selecting the free compute plan, and deploying the application.

The trained model (`model.pkl`), Flask application (`app.py`), prediction pipeline, and HTML template were included in the GitHub repository to ensure that all required components were available during deployment.

## 11.7 Deployment Results

The Flask application was successfully deployed on Render using the GitHub repository.

The deployment process completed successfully, and Render reported the application status as **Deploy succeeded | Live**.

The following results were observed:

- The application was successfully built and deployed.
- Gunicorn started successfully and listened on the port assigned by Render.
- The deployed application became accessible through a public web URL.
- The trained prediction model was available to the application through the prediction pipeline.

**Deployment Details**

| Item | Result |
|---|---|
| Deployment Platform | Render |
| Deployment Status | Successful |
| Deployment Branch | feature/data_preprocessing |
| Deployment Commit | 14eeb81 |
| Application Server | Gunicorn |
| Compute Plan | Free |

**Live Application URL:** [https://students-performance-prediction-bhie.onrender.com/]

The successful deployment demonstrates that the trained machine learning pipeline can be integrated into a web application and made accessible to users online.

## 11.8 Application Testing

After deployment, the web application was tested to verify that users could submit student information and receive predicted final grades.

### Prediction Testing

A sample student record was used to test the prediction functionality.

| Input Feature | Sample Value |
|---|---:|
| Study Time | 3 |
| Absences | 2 |
| First Period Grade (G1) | 12 |
| Second Period Grade (G2) | 14 |

The application processes the submitted information through the prediction pipeline and returns an estimated final mathematics grade (G3) on a scale of 0–20.

### Input Validation Testing

Server-side validation was implemented to ensure that users provide acceptable input values.

The following conditions were tested:

| Test Case | Expected Behaviour |
|---|---|
| Negative absences | Reject input |
| Study time outside the range 1–4 | Reject input |
| G1 greater than 20 | Reject input |
| G2 below 0 | Reject input |
| Missing required input | Display validation error |

These checks help prevent invalid information from being submitted to the prediction pipeline.

### Testing Outcome

The application was successfully deployed, and server-side validation was tested using Flask's test client during development.

The live application provides a web-based interface through which users can submit student information and obtain predicted academic performance.

## 11.9 Conclusion

This unit focused on developing a complete prediction pipeline and deploying the trained student performance prediction model as a web application.

The major activities completed included:

1. Developing a custom data class to collect student information.
2. Creating a prediction pipeline to load the saved machine learning model and generate predictions.
3. Integrating the prediction pipeline with a Flask web application.
4. Designing a user interface to allow users to submit student information.
5. Implementing server-side input validation and error handling.
6. Preparing project dependencies and deployment configuration.
7. Deploying the application successfully on Render.

The completed application allows users to enter student academic information and obtain an estimated final mathematics grade.

Overall, this unit demonstrates how a trained machine learning model can be transformed into an accessible web-based prediction system, completing the transition from model development to deployment.